<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-08c-agent-environments-task-spec-reward-and-reward-hacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 8 (continued) - Building an agent environment: task spec, sandbox, reward, and reward hacking

Chapter 8 evaluated agents that somebody else had set up. This lab builds the thing underneath: an **environment** in which an agent acts and is scored. It is the same object whether you use it to *evaluate* an agent, to collect trajectories, or to *train* one with reinforcement learning (the GRPO section of the post-training chapter consumed exactly this: a checker that returns a reward).

We build a small SWE-bench-style environment from real data: 164 hand-written programming problems from OpenAI's HumanEval (MIT licence). Each task is a Python function with a **bug we inject** and a test suite. The agent sees the code and a *visible* part of the tests, edits the code, and is scored on tests it never saw. Then we attack our own environment the way a clever agent would, and fix it.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded; not needed for the certificate. Runs on the free Colab CPU in a few minutes. No LLM and no API key: the agents here are deliberately simple programs, so that the *environment* is the thing under test.

*Original code written for this course. Data: HumanEval (OpenAI, MIT licence, https://github.com/openai/human-eval). Execution happens in a subprocess with a time limit; this is a teaching sandbox, not a security boundary (see the note in Part 2).*

In [ ]:
import ast, copy, gzip, hashlib, zlib, json, os, random, subprocess, sys, tempfile, time, urllib.request
import numpy as np
print(sys.version.split()[0], "| numpy", np.__version__)

## Part 1 - From a benchmark to a task specification

A task needs four things written down before any agent runs: **what the agent sees**, **what it may do**, **what counts as success**, and **what is hidden**. We load HumanEval, keep the problems whose tests are a flat list of `assert candidate(...) == expected` lines (so each assert is one independent check), inject one bug into the reference solution, and keep only tasks where the bug really breaks the tests.

In [ ]:
URL = "https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz"
raw = urllib.request.urlopen(URL, timeout=60).read()
print("downloaded", len(raw), "bytes, sha256", hashlib.sha256(raw).hexdigest()[:16])
problems = [json.loads(l) for l in gzip.decompress(raw).decode().splitlines()]
print(len(problems), "problems; keys:", list(problems[0].keys()))

def flat_asserts(test_src):
    # Return the assert lines of check() if every one is `assert candidate(...) == <expr>`; else None.
    tree = ast.parse(test_src)
    fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "check")
    out = []
    for st in fn.body:
        if isinstance(st, ast.Assert) and isinstance(st.test, ast.Constant):      # `assert True` placeholder
            continue
        ok = (isinstance(st, ast.Assert) and isinstance(st.test, ast.Compare) and len(st.test.ops) == 1
              and isinstance(st.test.ops[0], ast.Eq) and isinstance(st.test.left, ast.Call)
              and isinstance(st.test.left.func, ast.Name) and st.test.left.func.id == "candidate")
        if not ok: return None
        out.append(ast.unparse(st))
    return out

usable = []
for p in problems:
    a = flat_asserts(p["test"])
    if a and len(a) >= 5:
        usable.append((p, a))
print(len(usable), "problems have >= 5 flat equality asserts")

**The sandbox.** The agent's code is never run inside our own process. We write the program and the tests to a child Python process (`-I`, isolated mode), give it a wall-clock limit, and read one line of JSON back. A task that loops forever costs the agent a failed run, not a hung evaluation.

In [ ]:
RUNNER = '''
import sys, json
payload = json.load(sys.stdin)
res = {"load_error": None, "passed": []}
g = {}
try:
    exec(payload["source"], g)
    cand = g[payload["entry"]]
except BaseException as e:
    res["load_error"] = repr(e)[:160]
    print(json.dumps(res)); sys.exit()
for t in payload["tests"]:
    try:
        exec(t, dict(g, candidate=cand)); res["passed"].append(True)
    except BaseException:
        res["passed"].append(False)
print(json.dumps(res))
'''
RUNNER_PATH = os.path.join(tempfile.gettempdir(), "aibits_runner.py")
open(RUNNER_PATH, "w").write(RUNNER)

def _limits():                                          # POSIX only: cap memory and CPU seconds
    try:
        import resource
        resource.setrlimit(resource.RLIMIT_AS, (1 << 30, 1 << 30)); resource.setrlimit(resource.RLIMIT_CPU, (10, 10))
    except Exception:
        pass

_CACHE = {}
def run_tests(source, entry, tests, timeout=6):
    # -> list of booleans, one per test. Timeout or crash => all False. Results are cached: the same program on the same tests is deterministic.
    key = (source, entry, tuple(tests))
    if key not in _CACHE: _CACHE[key] = _run_tests(source, entry, tests, timeout)
    return list(_CACHE[key])

def _run_tests(source, entry, tests, timeout):
    try:
        p = subprocess.run([sys.executable, "-I", RUNNER_PATH], input=json.dumps({"source": source, "entry": entry, "tests": tests}),
                           capture_output=True, text=True, timeout=timeout, preexec_fn=_limits if os.name == "posix" else None)
        res = json.loads(p.stdout.strip().splitlines()[-1])
        return [False] * len(tests) if res["load_error"] else res["passed"]
    except Exception:
        return [False] * len(tests)

p0, a0 = usable[0]
ref0 = p0["prompt"] + p0["canonical_solution"]
print(p0["task_id"], "reference passes", sum(run_tests(ref0, p0["entry_point"], a0)), "of", len(a0), "tests")

**Injecting a bug.** We use four mutation operators, each the kind of slip people really make: flip a comparison (`<` to `<=`, `==` to `!=`), swap an arithmetic operator (`+` and `-`, `*` and `//`), nudge an integer constant by one, swap `and` with `or`. A mutant is kept only if it fails at least one of the task's own tests; otherwise there is no bug to find.

In [ ]:
CMP = {ast.Lt: ast.LtE, ast.LtE: ast.Lt, ast.Gt: ast.GtE, ast.GtE: ast.Gt, ast.Eq: ast.NotEq, ast.NotEq: ast.Eq}
ARITH = {ast.Add: ast.Sub, ast.Sub: ast.Add, ast.Mult: ast.FloorDiv, ast.FloorDiv: ast.Mult}
BOOL = {ast.And: ast.Or, ast.Or: ast.And}

def func_nodes(tree, entry):
    fn = next(n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == entry)
    return list(ast.walk(fn))

def sites(source, entry):
    # Every place a single mutation can be applied: (kind, node_index, variant)
    out = []
    for i, n in enumerate(func_nodes(ast.parse(source), entry)):
        if isinstance(n, ast.Compare) and len(n.ops) == 1 and type(n.ops[0]) in CMP: out.append(("cmp", i, 0))
        elif isinstance(n, ast.BinOp) and type(n.op) in ARITH: out.append(("arith", i, 0))
        elif isinstance(n, ast.BoolOp) and type(n.op) in BOOL: out.append(("bool", i, 0))
        elif isinstance(n, ast.Constant) and type(n.value) is int: out += [("const", i, 0), ("const", i, 1)]
    return out

def mutate(source, entry, site):
    kind, idx, var = site
    tree = ast.parse(source); n = func_nodes(tree, entry)[idx]
    if kind == "cmp": n.ops[0] = CMP[type(n.ops[0])]()
    elif kind == "arith": n.op = ARITH[type(n.op)]()
    elif kind == "bool": n.op = BOOL[type(n.op)]()
    else: n.value = n.value + (1 if var == 0 else -1)
    return ast.unparse(tree)

rng = random.Random(7)
TASKS = []
MAX_TASKS = 40                                                 # keep the lab fast; raise it for tighter estimates
for p, asserts in usable:
    if len(TASKS) >= MAX_TASKS: break
    ref = ast.unparse(ast.parse(p["prompt"] + p["canonical_solution"]))
    ss = sites(ref, p["entry_point"]); rng.shuffle(ss)
    for s in ss[:6]:                                         # try a few mutants, keep the first that really breaks something
        buggy = mutate(ref, p["entry_point"], s)
        passed = run_tests(buggy, p["entry_point"], asserts)
        if sum(passed) < len(asserts):
            TASKS.append(dict(id=p["task_id"], entry=p["entry_point"], reference=ref, buggy=buggy, bug_kind=s[0],
                              tests=asserts, visible=asserts[:2], hidden=asserts[2:]))
            break
print(len(TASKS), "tasks with an injected, test-breaking bug")
from collections import Counter
print("bug kinds:", dict(Counter(t["bug_kind"] for t in TASKS)))
t = TASKS[0]; print("\nexample", t["id"], "- bug kind:", t["bug_kind"], "| visible tests:", len(t["visible"]), "| hidden tests:", len(t["hidden"]))

## Part 2 - The environment (reset, step, reward)

The interface is the standard reinforcement-learning one (Gymnasium uses the same shape): `reset(task)` returns an observation; `step(action)` returns `(observation, reward, terminated, truncated, info)`. Our actions are `run_visible`, `edit` (replace the source), `edit_tests` (only offered when we deliberately leave the tests writable, Part 4), and `submit`. Every step costs 0.01, so wandering is mildly penalised; the real reward arrives on `submit`.

Two scorer designs are built in, because the choice is the whole point of Part 4: **`naive`** scores with whichever tests are currently in the workspace; **`held_out`** scores with the original test list, and the agent never sees the hidden tests.

In [ ]:
class FixBugEnv:
    STEP_COST = 0.01
    def __init__(self, tasks, scorer="held_out", tests_writable=False, max_steps=12, reward="fraction"):
        self.tasks, self.scorer, self.tests_writable, self.max_steps, self.reward_mode = tasks, scorer, tests_writable, max_steps, reward
    def reset(self, i):
        self.t = self.tasks[i]; self.source = self.t["buggy"]; self.workspace_tests = list(self.t["visible"]); self.steps = 0
        self.runs = 0; return self._obs(None), {"task_id": self.t["id"]}
    def _obs(self, last):
        return {"source": self.source, "tests": list(self.workspace_tests), "last": last, "steps_left": self.max_steps - self.steps}
    def _score(self):
        if self.scorer == "naive":                      # tests as they now stand in the workspace
            tests = self.workspace_tests
        else:                                           # the original suite, regardless of what the agent did to the workspace
            tests = self.t["tests"]
        passed = run_tests(self.source, self.t["entry"], tests)
        return (sum(passed) / len(tests)) if tests else 1.0
    def step(self, action):
        self.steps += 1; reward = -self.STEP_COST; terminated = False; info = {}
        kind = action["type"]
        if kind == "run_visible":
            self.runs += 1; passed = run_tests(self.source, self.t["entry"], self.workspace_tests); last = {"passed": sum(passed), "of": len(passed)}
        elif kind == "edit":
            self.source = action["source"]; last = "edited"
        elif kind == "edit_tests" and self.tests_writable:
            self.workspace_tests = list(action["tests"]); last = "tests edited"
        elif kind == "submit":
            s = self._score(); terminated = True; last = {"score": s}
            reward += (1.0 if s == 1.0 else 0.0) if self.reward_mode == "binary" else s
        else:
            last = "invalid action"
        truncated = (not terminated) and self.steps >= self.max_steps
        info["runs"] = self.runs
        return self._obs(last), reward, terminated, truncated, info
    def true_success(self):                             # ground truth, independent of the scorer in use
        return all(run_tests(self.source, self.t["entry"], self.t["tests"]))

def rollout(env, agent, i):
    obs, _ = env.reset(i); agent.start(env.t if hasattr(agent, "needs_task") else None)
    ret, done = 0.0, False
    while not done:
        obs, r, term, trunc, info = env.step(agent.act(obs)); ret += r; done = term or trunc
    return dict(ret=ret, true_success=env.true_success(), runs=info["runs"], steps=env.steps, submitted=term)

print("environment ready:", len(TASKS), "tasks")

## Part 3 - Test the environment before you test any agent

A broken environment gives confident, wrong numbers, so an environment gets its own tests. We need: the reference solution scores a full reward; submitting the untouched buggy code scores low; the same actions give the same reward twice; and an infinite loop ends in a failed run rather than a hang.

In [ ]:
class Scripted:
    def __init__(self, actions): self.actions = list(actions); self.k = 0
    def start(self, task=None): self.k = 0
    def act(self, obs):
        a = self.actions[min(self.k, len(self.actions) - 1)]; self.k += 1; return a

env = FixBugEnv(TASKS)
oracle_ret, noop_ret, repeat_equal = [], [], 0
for i in range(len(TASKS)):
    rets = []
    for _ in range(2):                                   # the same two actions, twice
        env.reset(i); _, r1, *_ = env.step({"type": "edit", "source": TASKS[i]["reference"]}); _, r2, *_ = env.step({"type": "submit"})
        rets.append(r1 + r2)
    oracle_ret.append(rets[0]); repeat_equal += abs(rets[0] - rets[1]) < 1e-9
    env.reset(i); _, rn, *_ = env.step({"type": "submit"}); noop_ret.append(rn)
print(f"reference solution: mean return {np.mean(oracle_ret):.3f}  (min {min(oracle_ret):.2f})")
print(f"submit the buggy code unchanged: mean return {np.mean(noop_ret):+.3f}  (fraction-scored: partial credit for the tests it still passes)")
print("the same actions gave the same return on a repeat:", repeat_equal, "of", len(TASKS))

loop_src = "def " + TASKS[0]["entry"] + "(*a, **k):\n    while True:\n        pass\n"
env.reset(0); env.step({"type": "edit", "source": loop_src}); t0 = time.time(); _, r, term, *_ = env.step({"type": "submit"})
print(f"infinite-loop submission: returned after {time.time()-t0:.1f}s with reward {r:+.2f} (terminated={term}); the environment did not hang")

**Read it.** The reference gets about +1 (minus the step costs of its two steps), the unchanged buggy code gets only partial credit, the same actions reproduce the same reward, and the infinite loop is contained. Those four checks are cheap and catch most environment bugs. Also look at *partial credit*: the buggy code still passes some of its tests, so a **fractional** reward pays a do-nothing agent roughly a third of the maximum (about 0.31 here). We come back to that in Part 5.

*Safety note.* A subprocess with a timeout and (on Linux) memory and CPU limits is enough for code you wrote. For code an LLM wrote, treat the sandbox as a security boundary: a container or a micro-VM with no network, no secrets mounted, a read-only filesystem apart from a scratch directory, and a non-root user. The security chapter's information-flow section explains why.

## Part 4 - Agents, and how they beat our environment

Five agents, none of them an LLM. Each is a stand-in for something a real agent does:

- **no-op**: submits immediately (the floor).
- **repair search**: enumerates single-edit repairs of the buggy code and submits the first one that passes the *visible* tests (the classic generate-and-validate repair loop).
- **hard-coder**: ignores the bug and writes a function that returns the expected outputs of the visible tests.
- **test-deleter**: when the tests are writable, deletes them. Absurd, and yet agents trained on a flawed environment do find this kind of move.
- **reference**: submits the correct solution (the ceiling; it checks the environment).

In [ ]:
def expected_pairs(visible):
    # parse `assert candidate(args) == expected` into {args-repr: expected value}
    table = {}
    for s in visible:
        node = ast.parse(s).body[0].test
        args = [ast.literal_eval(a) for a in node.left.args]
        table[repr(tuple(args))] = ast.literal_eval(node.comparators[0])
    return table

class NoOp:
    needs_task = False
    def start(self, task=None): pass
    def act(self, obs): return {"type": "submit"}

class Reference:
    needs_task = True
    def start(self, task): self.t = task; self.k = 0
    def act(self, obs):
        self.k += 1; return {"type": "edit", "source": self.t["reference"]} if self.k == 1 else {"type": "submit"}

class HardCoder:
    needs_task = True
    def start(self, task): self.t = task; self.k = 0
    def act(self, obs):
        self.k += 1
        if self.k == 1:
            tbl = expected_pairs(self.t["visible"])
            src = f"def {self.t['entry']}(*a):\n    return {tbl!r}[repr(a)]\n"
            return {"type": "edit", "source": src}
        return {"type": "submit"}

class TestDeleter:
    needs_task = False
    def start(self, task=None): self.k = 0
    def act(self, obs):
        self.k += 1; return {"type": "edit_tests", "tests": []} if self.k == 1 else {"type": "submit"}

class RepairSearch:
    needs_task = True
    def __init__(self, order=None, seed=0): self.order, self.seed = order, seed
    def start(self, task):
        self.t = task; self.k = 0; self.queue = None; self.rng = random.Random(self.seed * 1000 + zlib.crc32(task["id"].encode()) % 1000)      # stable across runs (Python's hash() is randomised)
    def _plan(self, obs):
        ss = sites(obs["source"], self.t["entry"]); self.rng.shuffle(ss)
        if self.order: ss.sort(key=lambda s: self.order.index(s[0]))      # stable: learned operator order first
        self.queue = [mutate(obs["source"], self.t["entry"], s) for s in ss[:30]]; self.pending = None
    def act(self, obs):
        if self.queue is None: self._plan(obs)
        last = obs["last"]
        if isinstance(last, dict) and "passed" in last and last["passed"] == last["of"]: return {"type": "submit"}
        if self.pending == "edited":
            self.pending = "ran"; return {"type": "run_visible"}
        if not self.queue: return {"type": "submit"}
        self.pending = "edited"; return {"type": "edit", "source": self.queue.pop(0)}

def evaluate(agent, env, idx):
    rows = [rollout(env, agent, i) for i in idx]
    return dict(reward=np.mean([r["ret"] for r in rows]), true=np.mean([r["true_success"] for r in rows]),
                runs=np.mean([r["runs"] for r in rows]), n=len(rows))

ALL = list(range(len(TASKS)))
print(f"{'agent':14}{'scorer':>10}{'tests':>10}{'mean reward':>14}{'TRUE success':>14}")
for name, agent, scorer, writable in [("no-op", NoOp(), "held_out", False), ("reference", Reference(), "held_out", False),
                                      ("hard-coder", HardCoder(), "held_out", False), ("repair search", RepairSearch(), "held_out", False),
                                      ("test-deleter", TestDeleter(), "naive", True), ("test-deleter", TestDeleter(), "held_out", True)]:
    e = FixBugEnv(TASKS, scorer=scorer, tests_writable=writable, reward="binary")
    r = evaluate(agent, e, ALL)
    print(f"{name:14}{scorer:>10}{'writable' if writable else 'fixed':>10}{r['reward']:>14.3f}{r['true']:>14.1%}")

**Read it.** Look at the two test-deleter rows. With the naive scorer and writable tests, an agent that deletes the tests earns the maximum reward while fixing nothing: the reward went up and the true success rate stayed where it was. That is **reward hacking** (also called specification gaming): the agent optimised the number we wrote down rather than the thing we meant. With the held-out scorer, the same move is worth nothing.

The hard-coder row shows the quieter version. It passes everything it can see and fails what it cannot, which is why the hidden tests exist. The repair-search row shows the honest middle: some patches that satisfy the visible tests are *plausible but wrong* (they overfit the two visible tests), a well-known phenomenon in automated program repair. Compare the repair search's visible-pass behaviour to its true success rate next.

In [ ]:
env = FixBugEnv(TASKS, scorer="held_out", reward="binary")
rows = [rollout(env, RepairSearch(), i) for i in ALL]
vis_ok = []
for i, r in enumerate(rows):
    env.reset(i); a = RepairSearch(); a.start(TASKS[i]); obs, _ = env.reset(i); done = False
    while not done:
        obs, rew, term, trunc, info = env.step(a.act(obs)); done = term or trunc
    vis_ok.append(all(run_tests(env.source, TASKS[i]["entry"], TASKS[i]["visible"])))
print(f"repair search over {len(TASKS)} tasks:")
print(f"  submitted code that passes ALL VISIBLE tests : {np.mean(vis_ok):.1%}")
print(f"  TRUE success (all original tests)            : {np.mean([r['true_success'] for r in rows]):.1%}")
print(f"  overfitting gap (visible-only pass, not truly fixed): {np.mean([v and not r['true_success'] for v, r in zip(vis_ok, rows)]):.1%} of tasks")

**Defences that worked here.** (1) Score with a **held-out** suite the agent cannot read or write. (2) Keep the tests **read-only** (or check their hash before scoring). (3) Report **visible versus hidden** pass rates side by side; a large gap is the alarm. (4) Run a **reference solution** and a **no-op** through the environment before every release of it. (5) Read some trajectories by hand: a number alone never reveals the test-deleting agent.

## Part 5 - Scoring design changes what you measure

Binary reward (1 only if everything passes) and fractional reward (share of tests passed) rank the same agents differently. We compute both on the no-op, hard-coder and repair-search agents.

In [ ]:
print(f"{'agent':16}{'binary reward':>16}{'fractional reward':>20}{'TRUE success':>16}")
for name, agent in [("no-op", NoOp()), ("hard-coder", HardCoder()), ("repair search", RepairSearch()), ("reference", Reference())]:
    out = {}
    for mode in ("binary", "fraction"):
        out[mode] = evaluate(agent, FixBugEnv(TASKS, reward=mode), ALL)
    print(f"{name:16}{out['binary']['reward']:>16.3f}{out['fraction']['reward']:>20.3f}{out['binary']['true']:>16.1%}")

**Read it.** Under binary reward the do-nothing and hard-coding agents score about zero and the repair search scores what it really achieved. Under fractional reward the do-nothing agent already earns about 0.31 and the hard-coder about 0.28, both with 0% true success, and the repair search's failures are also paid (0.63 against 0.48). A learner trained on the fractional signal can be rewarded for moving from 0.31 towards 0.6 on every task without ever fixing one. A *sparse* binary reward has the opposite problem: it gives a learner nothing to climb until it first succeeds by luck. Neither is "right". Choose by purpose: binary (or a threshold) to *evaluate* and to define success; shaped or fractional signals only when you are *training*, and then track the binary success rate alongside, so you can see whether the shaping is what the agent is learning to exploit.

## Part 6 - The environment as a training signal

To see the environment doing what an RL environment is for, we let a tiny learner improve from reward. The learner chooses which kind of edit to try first (`cmp`, `arith`, `const`, `bool`). It is an epsilon-greedy bandit: after each training episode it updates its estimate of how often each kind of edit led to a true fix on that kind of bug, and orders its search accordingly. Training tasks and test tasks are disjoint.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(TASKS)); train, test = list(idx[: len(idx)//2]), list(idx[len(idx)//2 :])
KINDS = ["cmp", "arith", "const", "bool"]
value = {k: 0.0 for k in KINDS}; count = {k: 0 for k in KINDS}
env = FixBugEnv(TASKS, reward="binary")                              # max_steps=12: an edit and a run cost a step each, so only about five candidates fit

curve = []
for ep in range(240):
    ti = int(train[ep % len(train)])
    explore = rng.random() < 0.2
    pick = KINDS[rng.integers(len(KINDS))] if explore else max(KINDS, key=lambda k: value[k])
    order = [pick] + [k for k in KINDS if k != pick]
    r = rollout(env, RepairSearch(order=order, seed=ep), ti)
    reward = 1.0 if r["true_success"] else 0.0
    count[pick] += 1; value[pick] += (reward - value[pick]) / count[pick]
    curve.append(reward)
print("estimated success when this kind of edit is tried first:", {k: f"{value[k]:.2f} (n={count[k]})" for k in KINDS})
print("training success rate by block of 60 episodes:", [round(float(np.mean(curve[i:i + 60])), 2) for i in range(0, 240, 60)])

learned = sorted(KINDS, key=lambda k: -value[k]); print("learned order:", learned)
def mean_over_seeds(order, seeds=range(1, 9)):
    res = [evaluate(RepairSearch(order=order, seed=sd), FixBugEnv(TASKS, reward="binary"), test) for sd in seeds]
    return np.mean([r["true"] for r in res]), np.mean([r["runs"] for r in res])
b_t, b_r = mean_over_seeds(None); l_t, l_r = mean_over_seeds(learned)
print(f"held-out tasks ({len(test)}), mean of 8 search seeds:")
print(f"  random order  -> true success {b_t:.1%}, visible-test runs per task {b_r:.2f}")
print(f"  bandit order  -> true success {l_t:.1%}, visible-test runs per task {l_r:.2f}")

**Read it, carefully.** The bandit's estimate for the kind it favoured is high on the training tasks, yet its order did not beat a random order on the held-out tasks. The reason is *credit assignment*: whether an episode succeeds depends mostly on **how hard the task is**, not on which operator was tried first, and the learner's estimates mix the two. A kind that happened to be picked on easy tasks looks good. The remedy used by the GRPO method from the post-training chapter is a **group-relative baseline**: sample several attempts at the *same* task, and credit each attempt with how far it is above or below the group's average for that task. Below, every training task is attempted with each of the four first choices (three search seeds each) and each choice is scored by its advantage over the task's own average.

In [ ]:
adv = {k: [] for k in KINDS}
for ti in train:
    rs = {k: np.mean([rollout(env, RepairSearch(order=[k] + [x for x in KINDS if x != k], seed=sd), int(ti))["true_success"] for sd in range(3)]) for k in KINDS}
    base_ti = np.mean(list(rs.values()))                                  # this task's own average: the group baseline
    for k in KINDS: adv[k].append(rs[k] - base_ti)
group_value = {k: float(np.mean(v)) for k, v in adv.items()}
print("mean advantage over each task's own average, by first-tried kind:", {k: round(v, 3) for k, v in group_value.items()})
grp_order = sorted(KINDS, key=lambda k: -group_value[k]); print("group-relative order:", grp_order)
g_t, g_r = mean_over_seeds(grp_order)
print(f"held-out tasks ({len(test)}), mean of 8 search seeds:")
print(f"  random order         -> true success {b_t:.1%}, visible-test runs per task {b_r:.2f}")
print(f"  bandit order         -> true success {l_t:.1%}, visible-test runs per task {l_r:.2f}")
print(f"  group-relative order -> true success {g_t:.1%}, visible-test runs per task {g_r:.2f}")

**Read it.** In the run shown here none of the three orders beat the random one on the held-out tasks (about 60% each), so this experiment gives a *null result*, and a null result is a result. The group-relative advantages at least point in an explicable direction: trying `const` edits first is *below* each task's own average, because a constant has two candidate edits (+1 and -1) and so `const` contributes the most candidate sites, which uses up the twelve-step budget before the right edit is reached. The effect (under 0.1) is far too small to separate on 20 test tasks. With an environment this cheap, the next move is not a cleverer learner but more tasks, until the interval for the difference is narrow enough to decide. Your numbers may differ slightly with your Python version, because the printed form of the code (and so the candidate order) can change.

**The bigger picture.** The learners are tiny and the data small (20 training tasks, 20 test tasks), so treat any difference as a hint, not a result; Chapter 8's statistics section says how to test it properly. The structure is the lesson. The environment turned a program's behaviour into a number; a learner used the number to change what it does; and held-out tasks tested whether what it learned transfers. Note that the budget of twelve steps is what made the *order* of attempts matter at all: with an unlimited budget every order eventually succeeds. Replace the bandit with a language model and the edit choice with a generated patch, add the GRPO update from the post-training chapter, and you have the shape of the training loops used to build coding agents; the environment you wrote above is the part that decides whether that loop learns to fix bugs or to delete tests.

## Part 7 - Checklist for an environment you will hand to someone else

1. **Task spec**: observation, action set, step limit, success condition, written down; versioned with the tasks.
2. **Sandbox**: time, memory, no network, no secrets, read-only everywhere the agent need not write.
3. **Scorer**: held-out checks; deterministic; cannot be influenced by the agent; binary success plus any shaped signal reported separately.
4. **Self-tests**: reference solution passes, no-op fails, rerun gives the same score, hangs are contained.
5. **Hack hunt**: read trajectories, compare visible and hidden scores, look for the agent that scores high by an unintended route.
6. **Documentation**: how to run it, what it does not measure, known failure modes, licence of the tasks and data (here MIT), and the version and hashes used for any published number.

## What to hand in

The Part 4 table for your own set of tasks, one new hack you invented and whether the held-out scorer stopped it, and a paragraph on whether you would use binary or fractional reward to *train* an agent on this environment and why.